# Gold Date Dimension

This notebook builds a continuous calendar dimension from the date and timestamp fields available in Silver orders and reviews.

**Grain:** One row per calendar date.

In [0]:
from pyspark.sql import functions as F

## 1. Define storage paths

In [0]:
SILVER_ORDERS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/orders"
)

SILVER_ORDER_REVIEWS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/order_reviews"
)

GOLD_DIM_DATES_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/dim_dates"
)

print(f"Orders source: {SILVER_ORDERS_PATH}")
print(f"Reviews source: {SILVER_ORDER_REVIEWS_PATH}")
print(f"Gold target: {GOLD_DIM_DATES_PATH}")

## 2. Read Silver inputs

In [0]:
silver_orders_df = (
    spark.read
    .format("delta")
    .load(SILVER_ORDERS_PATH)
)

silver_reviews_df = (
    spark.read
    .format("delta")
    .load(SILVER_ORDER_REVIEWS_PATH)
)

print(f"Silver order rows: {silver_orders_df.count():,}")
print(f"Silver review rows: {silver_reviews_df.count():,}")

## 3. Validate required date columns

In [0]:
required_order_columns = {
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
}

required_review_columns = {
    "review_creation_date",
    "review_answer_timestamp",
}

missing_order_columns = (
    required_order_columns - set(silver_orders_df.columns)
)

missing_review_columns = (
    required_review_columns - set(silver_reviews_df.columns)
)

if missing_order_columns:
    raise ValueError(
        "Silver orders is missing required date columns: "
        f"{sorted(missing_order_columns)}"
    )

if missing_review_columns:
    raise ValueError(
        "Silver reviews is missing required date columns: "
        f"{sorted(missing_review_columns)}"
    )

print("Required date column validation passed.")

## 4. Collect all relevant dates

In [0]:
order_dates_df = (
    silver_orders_df
    .select(
        F.explode(
            F.array(
                F.to_date("order_purchase_timestamp"),
                F.to_date("order_approved_at"),
                F.to_date("order_delivered_carrier_date"),
                F.to_date("order_delivered_customer_date"),
                F.to_date("order_estimated_delivery_date"),
            )
        ).alias("calendar_date")
    )
)

review_dates_df = (
    silver_reviews_df
    .select(
        F.explode(
            F.array(
                F.to_date("review_creation_date"),
                F.to_date("review_answer_timestamp"),
            )
        ).alias("calendar_date")
    )
)

all_dates_df = (
    order_dates_df
    .unionByName(review_dates_df)
    .filter(F.col("calendar_date").isNotNull())
)

date_bounds = (
    all_dates_df
    .agg(
        F.min("calendar_date").alias("minimum_date"),
        F.max("calendar_date").alias("maximum_date"),
    )
    .first()
)

minimum_date = date_bounds["minimum_date"]
maximum_date = date_bounds["maximum_date"]

if minimum_date is None or maximum_date is None:
    raise ValueError("No valid dates were found in the Silver inputs.")

print(f"Minimum date: {minimum_date}")
print(f"Maximum date: {maximum_date}")

## 5. Build continuous date dimension

In [0]:
dim_dates_df = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(minimum_date),
                F.lit(maximum_date),
                F.expr("INTERVAL 1 DAY"),
            )
        ).alias("date")
    )
    .withColumn(
        "date_key",
        F.date_format("date", "yyyyMMdd").cast("int"),
    )
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn(
        "day_of_week",
        ((F.dayofweek("date") + 5) % 7) + 1,
    )
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn(
        "is_weekend",
        F.col("day_of_week").isin(6, 7),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week",
        "day_name",
        "week_of_year",
        "is_weekend",
        "_gold_processed_at",
    )
)

display(dim_dates_df.limit(10))

## 6. Validate date dimension

In [0]:
dim_date_count = dim_dates_df.count()

expected_date_count = (
    maximum_date - minimum_date
).days + 1

duplicate_date_key_count = (
    dim_dates_df
    .groupBy("date_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_date_count = (
    dim_dates_df
    .groupBy("date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_date_key_count = (
    dim_dates_df
    .filter(F.col("date_key").isNull())
    .count()
)

if dim_date_count == 0:
    raise ValueError("Date dimension is empty.")

if dim_date_count != expected_date_count:
    raise ValueError(
        "Date dimension is not continuous. "
        f"Expected: {expected_date_count:,}, "
        f"Actual: {dim_date_count:,}"
    )

if duplicate_date_key_count > 0:
    raise ValueError(
        f"Date dimension contains "
        f"{duplicate_date_key_count:,} duplicate date keys."
    )

if duplicate_date_count > 0:
    raise ValueError(
        f"Date dimension contains "
        f"{duplicate_date_count:,} duplicate dates."
    )

if null_date_key_count > 0:
    raise ValueError(
        f"Date dimension contains "
        f"{null_date_key_count:,} null date keys."
    )

print(f"Date dimension rows: {dim_date_count:,}")
print("Date dimension continuity and grain validation passed.")

## 7. Write date dimension to Gold

In [0]:
(
    dim_dates_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_DIM_DATES_PATH)
)

print(f"Date dimension written to: {GOLD_DIM_DATES_PATH}")

## 8. Validate Gold output

In [0]:
written_dim_dates_df = (
    spark.read
    .format("delta")
    .load(GOLD_DIM_DATES_PATH)
)

written_date_count = written_dim_dates_df.count()

if written_date_count != dim_date_count:
    raise ValueError(
        "Gold date dimension write validation failed. "
        f"Expected: {dim_date_count:,}, "
        f"Written: {written_date_count:,}"
    )

print(f"Written date dimension rows: {written_date_count:,}")
print("Gold date dimension write validation passed.")

## 9. Inspect Gold date dimension

In [0]:
written_dim_dates_df.printSchema()

display(
    written_dim_dates_df
    .orderBy("date")
    .limit(20)
)